In [179]:
import pandas as pd
import numpy as np
import os
import duckdb

In [180]:
# Definindo a estrutura de caminhos relativos do projeto
BRONZE_DIR = "../data/bronze"
SILVER_DIR = "../data/silver"

In [181]:
def ler_nomes_arquivos_bronze():
    """
    Lê os nomes dos arquivos na pasta bronze e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(BRONZE_DIR) if os.path.isfile(os.path.join(BRONZE_DIR, f))]

In [182]:
lista_arquivos = ler_nomes_arquivos_bronze()

In [183]:
def transformar_csv_para_parquet(lista_arquivos):
    """
    Função para transformar um arquivo CSV em Parquet.
    
    Parâmetros:
    lista_arquivos (list): Lista de nomes dos arquivos CSV de entrada.
    """

    for arquivo in lista_arquivos:
        # Lendo o arquivo CSV
        df = pd.read_csv(os.path.join(BRONZE_DIR, arquivo))
        
        # Definindo o nome do arquivo Parquet de saída
        nome_arquivo_parquet = os.path.splitext(arquivo)[0] + '.parquet'
        
        # Salvando o DataFrame como Parquet
        df.to_parquet(os.path.join(SILVER_DIR, nome_arquivo_parquet), index=False)
        
        print(f"Arquivo {arquivo} transformado")


In [184]:
transformar_csv_para_parquet(lista_arquivos)

Arquivo br_bd_diretorios_brasil_cnae_2.csv transformado
Arquivo br_bd_diretorios_brasil_municipio.csv transformado
Arquivo br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv transformado


In [185]:
def ler_nomes_arquivos_silver():
    """
    Lê os nomes dos arquivos na pasta silver e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(SILVER_DIR) if os.path.isfile(os.path.join(SILVER_DIR, f))]

In [186]:
ler_nomes_arquivos_silver()

['br_bd_diretorios_brasil_cnae_2.parquet',
 'br_bd_diretorios_brasil_municipio.parquet',
 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet']

In [187]:
operacoes = pd.read_parquet(os.path.join(SILVER_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet'))

operacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 23483 entries, 0 to 23482
Data columns (total 39 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   razao_social_cliente                     23483 non-null  str    
 1   cnpj_cliente                             23474 non-null  float64
 2   descricao_projeto                        23483 non-null  str    
 3   sigla_uf                                 23483 non-null  str    
 4   nome_municipio                           23483 non-null  str    
 5   id_municipio                             15762 non-null  float64
 6   id_contrato                              23483 non-null  int64  
 7   data_contratacao                         23483 non-null  str    
 8   valor_contratado                         23483 non-null  float64
 9   valor_desembolsado                       23483 non-null  float64
 10  tipo_fonte_recursos                      22647 non-null  

#### Tratamento de valores nulos

In [188]:
def verificar_valores_nulos(df):
    """Verifica se um DataFrame contém valores nulos.

    Parâmetros:
        df (DataFrame): O DataFrame a ser verificado.

    Retorna:
        False | DataFrame: False se não houver nulos; DataFrame com análise se houver.
    """
    total_nulos = df.isnull().sum().sort_values(ascending=False)
    total_nulos = total_nulos[total_nulos > 0]

    if total_nulos.empty:
        return False

    total_nulos_percent = ((total_nulos / df.shape[0]) * 100).round(2)
    return pd.DataFrame({'Total Nulls': total_nulos, '%': total_nulos_percent})

In [189]:
verificar_valores_nulos(operacoes)

,Total Nulls,%
tipo_excepcionalidade,23265,99.07
cnpj_instituicao_financeira_credenciada,19412,82.66
nome_instituicao_financeira_credenciada,19412,82.66
id_municipio,7721,32.88
subclasse_cnae,5507,23.45
tipo_fonte_recursos,836,3.56
classe_cnae,497,2.12
situacao_contrato,268,1.14
grupo_cnae,255,1.09
cnpj_cliente,9,0.04


In [ ]:
#categorias existentes na coluna tipo_excepcionalidade
operacoes["tipo_excepcionalidade"].value_counts(dropna=False)

tipo_excepcionalidade
NaN                                                                                  23265
CONDIÇÕES DE CRÉDITO                                                                    81
CONDIÇÕES FINANCEIRAS E OPERACIONAIS                                                    68
COBRANÇA DE COMISSÕES                                                                   47
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /CONDIÇÕES DE CRÉDITO                              17
GARANTIAS                                                                                3
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /COBRANÇA DE COMISSÕES /CONDIÇÕES DE CRÉDITO        2
Name: count, dtype: int64

**Regra de negócio:** Se _tipo_excepcionalidade_
 está nulo, significa que a operação seguiu o fluxo padrão (sem exceção). Então, podemos realizar a classificação 0 = não, e 1 = sim.

In [191]:
operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)
operacoes[["tipo_excepcionalidade", "tem_excepcionalidade"]].drop_duplicates()

,tipo_excepcionalidade,tem_excepcionalidade
0,NaN,0
175,CONDIÇÕES DE CRÉDITO,1
934,CONDIÇÕES FINANCEIRAS E OPERACIONAIS,1
2762,CONDIÇÕES FINANCEIRAS E OPERACIONAIS /CONDIÇÕE...,1
7534,CONDIÇÕES FINANCEIRAS E OPERACIONAIS /COBRANÇA...,1
7535,COBRANÇA DE COMISSÕES,1
7903,GARANTIAS,1


In [192]:
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

In [193]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

nome_instituicao_financeira_credenciada    19412
cnpj_instituicao_financeira_credenciada    19412
id_municipio                                7721
subclasse_cnae                              5507
tipo_fonte_recursos                          836
classe_cnae                                  497
situacao_contrato                            268
grupo_cnae                                   255
cnpj_cliente                                   9
id_contrato                                    0
dtype: int64

**Regra de Négocio:** Se o CNPJ da instituição financeira credenciada estiver nulo, podemos entender que a operação foi realizada diretamente com o BNDES, sem intermediação de uma instituição financeira. Portanto, vamos preencher esses valores nulos com a string "OPERAÇÃO DIRETA".

In [194]:
operacoes["cnpj_instituicao_financeira_credenciada"] = operacoes["cnpj_instituicao_financeira_credenciada"].fillna("0000000000000.0")
operacoes["nome_instituicao_financeira_credenciada"] = operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
print(f"{operacoes[['cnpj_instituicao_financeira_credenciada']].dtypes}")

cnpj_instituicao_financeira_credenciada    object
dtype: object


In [195]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

id_municipio           7721
subclasse_cnae         5507
tipo_fonte_recursos     836
classe_cnae             497
situacao_contrato       268
grupo_cnae              255
cnpj_cliente              9
id_contrato               0
nome_municipio            0
sigla_uf                  0
dtype: int64

In [196]:
print(f"{operacoes[['cnpj_cliente']].dtypes}")

cnpj_cliente    float64
dtype: object


In [197]:
#preenchendo os valores nulos
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna("00000000000.0")
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

In [198]:
operacoes.head(5)

,razao_social_cliente,cnpj_cliente,descricao_projeto,sigla_uf,nome_municipio,id_municipio,id_contrato,data_contratacao,valor_contratado,valor_desembolsado,...,cnpj_instituicao_financeira_credenciada,tipo_garantia,situacao_contrato,secao_cnae,divisao_cnae,grupo_cnae,classe_cnae,subclasse_cnae,data_apuracao,tem_excepcionalidade
0,COPACOL-COOPERATIVA AGROINDUSTRIAL CONSOLATA,76093731000190.0,INVESTIMENTOS EM AMPLIACAO DA CAPACIDADE DE RE...,PR,SEM MUNICÍPIO,NÃO INFORMADO,9203361,2009-06-30,35900000.0,35900000.0,...,92816560000137.0,DEFINIDA PELO AGENTE FINANCEIRO,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04,0
1,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,PR,MATELANDIA,4115606.0,9203571,2009-06-29,66998000.0,66998000.0,...,0000000000000.0,REAL,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04,0
2,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,MS,SEM MUNICÍPIO,NÃO INFORMADO,9203581,2009-06-30,20897000.0,20897000.0,...,92816560000137.0,DEFINIDA PELO AGENTE FINANCEIRO,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04,0
3,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204441,2009-06-30,37000000.0,37000000.0,...,92816560000137.0,DEFINIDA PELO AGENTE FINANCEIRO,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04,0
4,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204451,2009-10-09,50000000.0,50000000.0,...,92816560000137.0,DEFINIDA PELO AGENTE FINANCEIRO,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04,0


In [199]:
#removendo colunas cnae desnecessárias
operacoes = operacoes.drop(columns=["classe_cnae","subclasse_cnae","grupo_cnae","divisao_cnae","secao_cnae"])

In [200]:
verificar_valores_nulos(operacoes)

False